# Nexus — Forecast Accuracy & Error Analysis
## Objective 1 — Understanding Where Revenue Is Lost and Why
> Forecast error analysis, over/under-forecasting rates, premium store impact,
> promotion amplification effect — feeds the Forecast & Model dashboard page.

## Section 0 — Imports & Config

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, gc
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

DATA_DIR   = Path('/Users/shashi/retail-ai-project/data/raw/output/csv')
OUTPUT_DIR = Path('./nexus_forecast_accuracy_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

NOW = datetime.now()
print(f'Forecast Accuracy Analysis — {NOW.strftime("%B %d, %Y %I:%M %p")}')


Forecast Accuracy Analysis — April 25, 2026 01:22 PM


## Section 1 — Load Demand Forecasts vs Actuals

In [2]:
stores   = pd.read_csv(DATA_DIR / 'stores.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
promos   = pd.read_csv(DATA_DIR / 'promotions.csv',
                        parse_dates=['start_date','end_date'])

# Load demand forecasts
print('Loading demand forecasts...')
forecast = pd.read_csv(DATA_DIR / 'demand_forecasts.csv',
                        parse_dates=['forecast_date'])

# Load actual sales
print('Loading sales...')
chunks = []
for chunk in pd.read_csv(DATA_DIR / 'sales_transactions.csv',
                          parse_dates=['sale_date'], chunksize=500_000):
    grp = (chunk.groupby(['store_id','sku_id','sale_date'])
               .agg(actual_units=('units_sold','sum'))
               .reset_index())
    chunks.append(grp)
actuals = pd.concat(chunks).groupby(['store_id','sku_id','sale_date']).sum().reset_index()
del chunks; gc.collect()

# Merge forecast with actuals
df = forecast.merge(
    actuals, left_on=['store_id','sku_id','forecast_date'],
    right_on=['store_id','sku_id','sale_date'], how='inner')

df = df.merge(stores[['store_id','region','store_format','foot_traffic_tier']],
              on='store_id', how='left')
df = df.merge(products[['sku_id','category','unit_price']], on='sku_id', how='left')

print(f'Forecast-actual pairs: {len(df):,}')
print(f'Forecast columns: {forecast.columns.tolist()}')


Loading demand forecasts...
Loading sales...
Forecast-actual pairs: 14,272,324
Forecast columns: ['forecast_id', 'store_id', 'sku_id', 'forecast_date', 'forecast_units', 'forecast_method', 'created_at', 'lower_bound_90', 'upper_bound_90']


## Section 2 — Core Error Metrics

In [3]:
# Determine forecast column name
fc_col = 'forecast_units' if 'forecast_units' in df.columns \
         else 'predicted_units' if 'predicted_units' in df.columns \
         else df.columns[3]  # fallback
print(f'Using forecast column: {fc_col}')

df['error']       = df[fc_col] - df['actual_units']
df['abs_error']   = df['error'].abs()
df['pct_error']   = (df['error'] / df['actual_units'].replace(0, np.nan)).fillna(0)
df['abs_pct_error']= df['pct_error'].abs()
df['is_over']     = (df['error'] > 0).astype(int)   # over-forecast
df['is_under']    = (df['error'] < 0).astype(int)   # under-forecast
df['revenue_at_risk'] = df['abs_error'] * df['unit_price'].fillna(0)

# WAPE
wape = df['abs_error'].sum() / df['actual_units'].replace(0,np.nan).sum() * 100

# Over and under rates
over_rate  = df['is_over'].mean() * 100
under_rate = df['is_under'].mean() * 100

print('='*52)
print('  FORECAST ACCURACY SUMMARY')
print('='*52)
print(f'  WAPE              : {wape:.2f}%')
print(f'  Over-forecast rate: {over_rate:.2f}%')
print(f'  Under-forecast rate: {under_rate:.2f}%')
print(f'  Total revenue at risk: ${df["revenue_at_risk"].sum():,.0f}')
print('='*52)


Using forecast column: forecast_units
  FORECAST ACCURACY SUMMARY
  WAPE              : 32.05%
  Over-forecast rate: 52.23%
  Under-forecast rate: 43.95%
  Total revenue at risk: $1,086,249,912


## Section 3 — Breakdown by Store Format, Region, Category

In [4]:
def accuracy_breakdown(group_col, df=df):
    return (
        df.groupby(group_col)
        .agg(
            wape             = ('abs_pct_error',  'mean'),
            over_forecast_rate=('is_over',         'mean'),
            under_forecast_rate=('is_under',        'mean'),
            revenue_at_risk  = ('revenue_at_risk', 'sum'),
            avg_shortage     = ('abs_error',       'mean'),
            total_pairs      = ('actual_units',    'count'),
        )
        .reset_index()
        .sort_values('revenue_at_risk', ascending=False)
    )

by_format   = accuracy_breakdown('store_format')
by_region   = accuracy_breakdown('region')
by_category = accuracy_breakdown('category')
by_traffic  = accuracy_breakdown('foot_traffic_tier')

print('BY STORE FORMAT:')
print(by_format[['store_format','wape','over_forecast_rate',
                  'under_forecast_rate','revenue_at_risk']].to_string(index=False))
print()
print('BY CATEGORY (top 10):')
print(by_category[['category','wape','revenue_at_risk']].head(10).to_string(index=False))


BY STORE FORMAT:
store_format     wape  over_forecast_rate  under_forecast_rate  revenue_at_risk
 Convenience 0.487914            0.524078             0.438595     2.364153e+08
   Wholesale 0.498825            0.521954             0.439167     2.355302e+08
 Hypermarket 0.499461            0.520512             0.441011     2.332418e+08
    Discount 0.500029            0.521023             0.440090     2.095392e+08
 Supermarket 0.494936            0.524639             0.438103     1.715235e+08

BY CATEGORY (top 10):
       category     wape  revenue_at_risk
      Beverages 0.495871     1.328021e+08
         Snacks 0.490889     1.107198e+08
   Dairy & Eggs 0.496829     1.051970e+08
   Frozen Foods 0.498979     9.654786e+07
  Personal Care 0.495490     9.570173e+07
         Bakery 0.498327     8.466907e+07
 Meat & Seafood 0.494210     8.218066e+07
      Household 0.500414     8.036464e+07
Canned & Pantry 0.496023     7.500742e+07
        Produce 0.492153     7.224862e+07


## Section 4 — Promotion Amplification Effect

In [5]:
# Does promotion cause larger forecast errors?
if 'is_promoted' in df.columns:
    promo_comp = df.groupby('is_promoted').agg(
        wape            = ('abs_pct_error', 'mean'),
        avg_abs_error   = ('abs_error',     'mean'),
        revenue_at_risk = ('revenue_at_risk','sum'),
        count           = ('actual_units',   'count'),
    ).reset_index()
    promo_comp['label'] = promo_comp['is_promoted'].map({True:'On Promotion', False:'Normal'})

    promo_wape   = promo_comp[promo_comp['is_promoted']==True]['wape'].values[0] * 100
    normal_wape  = promo_comp[promo_comp['is_promoted']==False]['wape'].values[0] * 100
    amplification= promo_wape / max(normal_wape, 0.001)

    print(f'Normal WAPE     : {normal_wape:.1f}%')
    print(f'Promo WAPE      : {promo_wape:.1f}%')
    print(f'Amplification   : {amplification:.1f}x')
    print(promo_comp[['label','wape','avg_abs_error','revenue_at_risk','count']].to_string(index=False))
else:
    print('No promotion column found — skipping promotion analysis')
    amplification = 1.0


No promotion column found — skipping promotion analysis


## Section 5 — Monthly & Seasonal Accuracy

In [6]:
df['month'] = df['forecast_date'].dt.month

monthly_acc = (
    df.groupby('month')
    .agg(
        wape            = ('abs_pct_error',  'mean'),
        revenue_at_risk = ('revenue_at_risk','sum'),
        avg_shortage    = ('abs_error',      'mean'),
    )
    .reset_index()
)
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_acc['month_name'] = monthly_acc['month'].map(month_names)

print('MONTHLY FORECAST ACCURACY:')
print(monthly_acc[['month_name','wape','revenue_at_risk']].to_string(index=False))


MONTHLY FORECAST ACCURACY:
month_name     wape  revenue_at_risk
       Jan 0.674826     7.653582e+07
       Feb 0.506956     7.666891e+07
       Mar 0.495957     7.772323e+07
       Apr 0.497039     8.926240e+07
       May 0.488949     9.158652e+07
       Jun 0.479318     8.351988e+07
       Jul 0.477629     1.080741e+08
       Aug 0.485208     8.655410e+07
       Sep 0.517312     8.328400e+07
       Oct 0.483062     1.036507e+08
       Nov 0.467127     9.018699e+07
       Dec 0.443466     1.192033e+08


## Section 6 — Visualizations & Export

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Forecast Accuracy Analysis — {NOW.strftime("%B %d, %Y")}',
             fontsize=13, fontweight='bold', color='white')
fig.patch.set_facecolor('#0D1B2A')
for ax in axes.flat:
    ax.set_facecolor('#132238')
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#2D4A6A')
    ax.spines['left'].set_color('#2D4A6A')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# P1: WAPE by category
top10 = by_category.nlargest(10, 'wape')
axes[0,0].barh(top10['category'], top10['wape']*100, color='#EF4444', alpha=0.85)
axes[0,0].set_title('WAPE by Category (worst)', color='white', fontweight='bold')
axes[0,0].set_xlabel('WAPE %', color='#94A3B8')

# P2: Monthly WAPE
colors2 = ['#EF4444' if v > monthly_acc['wape'].mean()*1.2 else '#0D9488'
            for v in monthly_acc['wape']]
axes[0,1].bar(monthly_acc['month_name'], monthly_acc['wape']*100, color=colors2)
axes[0,1].axhline(monthly_acc['wape'].mean()*100, color='white',
                  linestyle='--', linewidth=1, label='avg')
axes[0,1].set_title('WAPE by Month', color='white', fontweight='bold')
axes[0,1].set_ylabel('WAPE %', color='#94A3B8')
axes[0,1].legend(labelcolor='white', facecolor='#132238')

# P3: Over vs under by store format
x3 = np.arange(len(by_format))
axes[1,0].bar(x3-0.2, by_format['over_forecast_rate']*100,  0.35,
              label='Over-forecast', color='#F59E0B', alpha=0.85)
axes[1,0].bar(x3+0.2, by_format['under_forecast_rate']*100, 0.35,
              label='Under-forecast', color='#EF4444', alpha=0.85)
axes[1,0].set_xticks(x3)
axes[1,0].set_xticklabels(by_format['store_format'], rotation=20, ha='right', color='white')
axes[1,0].set_title('Over vs Under-Forecast by Format', color='white', fontweight='bold')
axes[1,0].set_ylabel('%', color='#94A3B8')
axes[1,0].legend(labelcolor='white', facecolor='#132238')

# P4: Revenue at risk by region
axes[1,1].barh(by_region['region'], by_region['revenue_at_risk'], color='#8B5CF6', alpha=0.85)
axes[1,1].set_title('Revenue at Risk by Region', color='white', fontweight='bold')
axes[1,1].set_xlabel('Revenue at Risk ($)', color='#94A3B8')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'forecast_accuracy.png',
            bbox_inches='tight', dpi=130, facecolor='#0D1B2A')
plt.show()

# ── Export all
by_format.to_csv(OUTPUT_DIR   / 'accuracy_by_format.csv',   index=False)
by_region.to_csv(OUTPUT_DIR   / 'accuracy_by_region.csv',   index=False)
by_category.to_csv(OUTPUT_DIR / 'accuracy_by_category.csv', index=False)
by_traffic.to_csv(OUTPUT_DIR  / 'accuracy_by_traffic.csv',  index=False)
monthly_acc.to_csv(OUTPUT_DIR / 'accuracy_monthly.csv',     index=False)

summary = pd.DataFrame([{
    'wape': round(wape,2), 'over_forecast_rate': round(over_rate,2),
    'under_forecast_rate': round(under_rate,2),
    'total_revenue_at_risk': round(df['revenue_at_risk'].sum(),2),
    'promo_amplification': round(amplification,2),
    'report_time': NOW.strftime('%Y-%m-%d %H:%M:%S'),
}])
summary.to_csv(OUTPUT_DIR / 'accuracy_summary.csv', index=False)

print('All outputs saved to nexus_forecast_accuracy_outputs/')
print(f'  wape={wape:.2f}%  over={over_rate:.1f}%  under={under_rate:.1f}%')


All outputs saved to nexus_forecast_accuracy_outputs/
  wape=32.05%  over=52.2%  under=43.9%
